## All sorts of techniques to speed up decoders-only and encoder-decoders


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TrainingArguments
import torch
import torch.nn as nn
import torch.nn.functional as F
import peft


In [3]:
torch.cuda.memory_allocated()

0

### Key/Value Caching

In [7]:
model_id = "facebook/opt-125m"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
prompt = "Once upon a time there lived"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
for use_cache in (True, False):
    print(f"{use_cache=}")
    %time model.generate(**inputs, max_new_tokens=500, do_sample=False, use_cache=use_cache)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

use_cache=True
CPU times: user 730 ms, sys: 0 ns, total: 730 ms
Wall time: 733 ms
use_cache=False
CPU times: user 886 ms, sys: 0 ns, total: 886 ms
Wall time: 890 ms


### Speculative Decoding


In [5]:
target_model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m", device_map='auto')

draft_model = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", device_map="auto")

tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")
prompt = "Once upon a time there lived"

inputs = tokenizer(prompt, return_tensors="pt").to(target_model.device)
outputs = target_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=1, assistant_model=draft_model)
results = tokenizer.decode(outputs[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  663MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  662MB            

... (9 outputs left out)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'use_cache', 'max_new_tokens', 'min_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
results

'Once upon a time there lived a rich child who would rather eat vegetables and eat a steak than play with the world…'

In [2]:
model_id = "google/bigbird-roberta-base"
pipeline = pipeline(task="fill-mask", model=model_id)
pipeline("She took some [MASK] medicine as she was feeling ill")

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  513MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] BigBirdForMaskedLM LOAD REPORT from: google/bigbird-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

... (3 outputs left out)

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

[transformers] Attention type 'block_sparse' is not possible if sequence_length: 13 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...


[{'score': 0.09868315607309341,
  'token': 42004,
  'token_str': 'emo',
  'sequence': 'She took some emo medicine as she was feeling ill'},
 {'score': 0.06327230483293533,
  'token': 47220,
  'token_str': 'roman',
  'sequence': 'She took some roman medicine as she was feeling ill'},
 {'score': 0.04653396084904671,
  'token': 439,
  'token_str': "'s",
  'sequence': "She took some 's medicine as she was feeling ill"},
 {'score': 0.04504694044589996,
  'token': 46684,
  'token_str': 'bp',
  'sequence': 'She took some bp medicine as she was feeling ill'},
 {'score': 0.039834048599004745,
  'token': 100,
  'token_str': '<unk>',
  'sequence': 'She took some  medicine as she was feeling ill'}]

## LSH implemnted for Approximate Attention

In [38]:
def angular_lsh(vectors, k):
    R= torch.randn(vectors.size(-1), k//2, device = vectors.device)
    normalized_vectors = F.normalize(vectors, p=2.0, dim =1)
    V_proj = normalized_vectors @ R
    V_concat = torch.cat([V_proj, -V_proj], dim=1)
    return torch.argmax(V_concat, dim=1)


In [39]:
torch.manual_seed(42)
vectors = torch.rand(16, 512)
angular_lsh(vectors, k=4)

tensor([2, 2, 0, 3, 0, 2, 2, 2, 2, 1, 1, 3, 3, 1, 2, 1])

## Performer

In [40]:
torch.manual_seed(42)
d, m = 64, 1024
W = torch.randn(d, m)
X = torch.randn(5, d) / d ** 0.5
R = torch.exp(X @ W)
print(R.mean(axis=-1))
print(torch.exp(0.5 * (X.norm(dim=-1)**2)))

tensor([1.5851, 1.6703, 1.6728, 1.9729, 1.7934])
tensor([1.6516, 1.7578, 1.6594, 1.8050, 1.7751])


In [41]:
def phi(X, W, dim_subtract_max=(-2, -1)):
    squared_norms = X.square().sum(dim=-1, keepdim=True)
    X_proj = X @ W
    max_vals = X_proj.amax(dim=dim_subtract_max, keepdim=True)
    return torch.exp(X_proj - max_vals - squared_norms / 2) / W.size(-1) ** 0.5

In [42]:
torch.manual_seed(42)
batch_size = 32
d_model = 512
n_heads = 8
Lq = 200
Lk = 100
m = 256
d_head = d_model // n_heads
W = torch.randn(n_heads, d_head, m)
Q = torch.randn(batch_size, n_heads, Lq, d_head)
K = torch.randn(batch_size, n_heads, Lk, d_head)
scale = 1 / d_head ** 0.5
Qp = phi(Q * scale ** 0.5, W, dim_subtract_max=-1)
Kp = phi(K * scale ** 0.5, W)
A = Qp @ Kp.transpose(-2, -1)
D = A.sum(dim=-1, keepdim=True)
result = A / (D + 1e-6)
expected = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)
rmse = F.mse_loss(result, expected) ** 0.5
rmse



tensor(0.0171)

In [44]:
class FavorAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_features):
        super().__init__()
        self.d_head = d_model // n_heads
        W = torch.randn(n_heads, self.d_head, n_features)  # h, d, m
        W = orthogonalize(W)
        self.register_buffer("W", W)

    def forward(self, Q, K, V):
        scale = self.d_head ** -0.25
        Qp = phi(Q * scale, self.W, dim_subtract_max=-1)
        Kp = phi(K * scale, self.W)
        D = Qp @ Kp.sum(dim=-2).unsqueeze(-1)  # B, h, Lq, 1
        Kp_T_V = Kp.transpose(-2, -1) @ V  # B, h, m, d
        return (Qp @ Kp_T_V) / (D + 1e-6)

In [45]:
def orthogonalize(W):
    d_head = W.size(-2)
    W_orth = torch.cat([torch.linalg.qr(W_chunk)[0]
                        for W_chunk in W.split(d_head, dim=-1)], dim=-1)
    return W_orth * d_head ** 0.5

## Grouped-query attention

In [47]:
batch_size, Lq, Lk, d_head =  32, 100, 90, 64
n_heads = 8
n_groups = 2
query = torch.randn(batch_size, n_groups, Lq, d_head)
key = torch.randn(batch_size, n_groups, Lk, d_head)
value = torch.randn(batch_size, n_groups, Lk, d_head)
attn = F.scaled_dot_product_attention(query, key, value,enable_gqa=True)
attn

tensor([[[[ 4.1962e-02,  1.7605e-01, -2.1073e-01,  ..., -3.1237e-02,
            2.6236e-01,  1.3001e-01],
          [ 2.8910e-03,  1.6124e-01,  1.7032e-01,  ...,  8.9749e-02,
            2.0786e-01,  3.4289e-02],
          [-1.9757e-01,  4.3499e-01,  5.6606e-01,  ...,  2.3202e-02,
            5.7778e-01,  4.0929e-01],
          ...,
          [-1.2020e-01, -2.1992e-02,  3.5902e-01,  ...,  1.8838e-01,
            1.3402e-01,  1.5028e-01],
          [ 8.2173e-02,  2.8687e-01,  6.8689e-02,  ..., -1.2477e-01,
           -8.9422e-02,  2.1003e-01],
          [-2.8483e-01,  1.1404e-01,  2.3932e-01,  ...,  4.4938e-01,
            4.0330e-02, -3.0112e-02]],

         [[-2.3424e-02,  2.0818e-01, -2.9174e-02,  ..., -3.9716e-03,
           -3.7505e-02,  4.9925e-02],
          [-1.9559e-01,  2.2398e-02, -1.2789e-01,  ..., -1.1355e-01,
            5.4918e-02, -1.3074e-01],
          [-1.2808e-01,  2.6941e-01, -1.7077e-01,  ...,  1.6034e-02,
           -8.4854e-03,  8.1936e-02],
          ...,
     

### Flash Attention Implementation

In [14]:
from sympy import O


def flash_attention(Q, K, V, block_size_q, block_size_k):
    Lq, d = Q.shape
    Lk, _ = K.shape
    O = torch.zeros_like(Q)
    scale = d ** -0.5

    for i_start in range(0, Lq, block_size_q):
        i_end = min(i_start + block_size_q, Lq)
        Q_block = Q[i_start:i_end]
        O_block = torch.zeros(block_size_q, d, device = Q.device)
        l_block = torch.zeros(block_size_q, 1 , device=Q.device)
        m_block = -torch.inf * torch.ones(block_size_q, 1, device=Q.device)

        for j_start in range(0, Lk, block_size_k):
            j_end = min(j_start + block_size_k, Lk)
            K_block = K[j_start:j_end]
            V_block = V[j_start:j_end]

            S_ij = Q_block @ K_block.T * scale

            m_ij_new, _ = torch.max(S_ij, dim=1, keepdim=True)
            m_block_new = torch.maximum(m_block, m_ij_new)

            P_ij = torch.exp(S_ij - m_block_new)
            correction_factor = torch.exp(m_block - m_block_new)

            l_block_new = ((l_block * correction_factor)
                           + torch.sum(P_ij, dim=1, keepdim=True))
            O_block = (O_block * correction_factor) + (P_ij @ V_block)

            l_block = l_block_new
            m_block = m_block_new

        O[i_start:i_end]= O_block / l_block
    return O

In [15]:
torch.manual_seed(42)
block_size_q, block_size_k = 64,64
Lq, Lk, d = 1280, 1152, 512
Q = torch.randn(Lq, d)
K = torch.randn(Lk, d)
V = torch.randn(Lk, d)

In [16]:
R1 = flash_attention(Q, K, V, block_size_q, block_size_k)
R2 =  F.scaled_dot_product_attention(Q, K, V)

In [17]:
mse = F.mse_loss(R1, R2)
mse

tensor(8.0208e-16)

## LoRA - Low Rank Adapter

In [7]:
model_id = "EleutherAI/gpt-neo-125m"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype=torch.float16)
lora_config = peft.LoraConfig(r=8,lora_alpha=32,  target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM", lora_dropout=0.05)
peft_model = peft.get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 294,912 || all params: 125,493,504 || trainable%: 0.2350


In [ ]:
merged_model = peft_model.merge_and_unload()

## Gradient Checkpoints in Transformers

In [ ]:
from transformers import TrainingArguments


args = TrainingArguments(output_dir="./model", num_train_epochs=3, per_device_train_batch_size=8, gradient_checkpointing=True, fp16=True)

## Gradient Accumulation

In [8]:
model = nn.Linear(10,1).to("cuda")
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)
criterion = nn.MSELoss()
data_loader = [(torch.randn(8, 10), torch.randn(8, 1)) for _ in range(100)]
model.train()

accumulation_steps = 4
optimizer.zero_grad()
for batch_index, (X_batch, y_batch) in enumerate(data_loader):
    X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
    y_pred = model(X_batch)
    loss = criterion(y_pred, y_batch)
    loss = loss / accumulation_steps
    loss.backward()
    if (batch_index + 1 ) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

## GPU Paralelism

In [ ]:
dp_model = torch.nn.DataParallel(model, device_ids=[0,1,2])